In [11]:
from pathlib import Path
import sys
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [12]:
import pandas as pd

from RADAR.static_data.anomaly_dataset_utils import (
    build_har_anomaly_dataset,
    build_kddcup99_anomaly_dataset,
    build_loaded_uci_anomaly_dataset,
    load_dataset_silently,
)

def benchmark_summary(benchmark):
    return pd.Series(
        {
            "name": benchmark["name"],
            "n_samples": benchmark["n_samples"],
            "n_features": benchmark["n_features"],
            "train_normals": benchmark["train_normals"],
            "test_normals": benchmark["test_normals"],
            "test_anomalies": benchmark["test_anomalies"],
            "original_positive_ratio": round(benchmark["original_positive_ratio"], 4),
            "benchmark_test_positive_ratio": round(benchmark["benchmark_test_positive_ratio"], 4),
        }
    )

In [13]:
dataset_name = "spambase"
X_raw, y_raw = load_dataset_silently(dataset_name)

print(f"{dataset_name}: X={X_raw.shape}, y={y_raw.shape}")
pd.Series(y_raw).value_counts().sort_index().rename("count")

spambase: X=(4601, 57), y=(4601,)


0    2788
1    1813
Name: count, dtype: int64

### Cargar un dataset UCI sin imprimir metadatos
`load_dataset_silently` devuelve `X` e `y` sin el texto extra que imprime `global_load`, ideal para ejemplos y notebooks.

In [14]:
dataset_name = "arrhythmia"
X_raw, y_raw = load_dataset_silently(dataset_name)

print(f"{dataset_name}: X={X_raw.shape}, y={y_raw.shape}")
pd.Series(y_raw).value_counts().sort_index().rename("count")

arrhythmia: X=(452, 279), y=(452,)


1     245
2      44
3      15
4      15
5      13
6      25
7       3
8       2
9       9
10     50
14      4
15      5
16     22
Name: count, dtype: int64

### Convertir un dataset UCI en benchmark de anomalías
`build_loaded_uci_anomaly_dataset` hace el split, binariza etiquetas, imputa, escala y devuelve listas las matrices de entrenamiento y evaluación.

In [ ]:
spambase_benchmark = build_loaded_uci_anomaly_dataset(
    "spambase",
    normal_label=0,
    target_test_contamination=0.1,
    random_state=42,
)

benchmark_summary(spambase_benchmark)

### Dataset especial: KDD Cup 99
Este helper encapsula la carga especial de KDD, convierte las clases a normal/anómalo y devuelve el benchmark ya preprocesado.

In [16]:
kdd_benchmark = build_kddcup99_anomaly_dataset(
    normal_label="normal",
    target_test_contamination=0.1,
    random_state=42,
    max_train_normals=10_000,
    max_test_size=5_000,
)

benchmark_summary(kdd_benchmark)

name                             kddcup99
n_samples                           20000
n_features                            106
train_normals                        3150
test_normals                          788
test_anomalies                         87
original_positive_ratio            0.8031
benchmark_test_positive_ratio      0.0994
dtype: object

### Dataset especial: Human Activity Recognition
Para HAR también hay un helper dedicado que parte de las 6 actividades y deja definido qué etiquetas se consideran normales.

In [17]:
har_benchmark = build_har_anomaly_dataset(
    normal_labels=[1, 2, 3],
    target_test_contamination=0.1,
    random_state=42,
    max_train_normals=5_000,
    max_test_size=3_000,
)

benchmark_summary(har_benchmark)

name                             human_activity_recognition
n_samples                                             10299
n_features                                              561
train_normals                                          3738
test_normals                                            934
test_anomalies                                          103
original_positive_ratio                              0.5464
benchmark_test_positive_ratio                        0.0993
dtype: object

In [18]:
summary_table = pd.DataFrame(
    [
        benchmark_summary(spambase_benchmark),
        benchmark_summary(kdd_benchmark),
        benchmark_summary(har_benchmark),
    ]
).reset_index(drop=True)

summary_table

,name,n_samples,n_features,train_normals,test_normals,test_anomalies,original_positive_ratio,benchmark_test_positive_ratio
0,spambase,4601,57,2230,558,62,0.3940,0.1000
1,kddcup99,20000,106,3150,788,87,0.8031,0.0994
2,human_activity_recognition,10299,561,3738,934,103,0.5464,0.0993


### Qué devuelve cada helper
Todos devuelven un diccionario con los arrays listos para modelado: `X_train`, `X_test`, `y_test` y métricas de resumen del benchmark.

In [19]:
{
    "spambase_keys": sorted(spambase_benchmark.keys()),
    "kdd_keys": sorted(kdd_benchmark.keys()),
    "har_keys": sorted(har_benchmark.keys()),
}

{'spambase_keys': ['X_test',
  'X_train',
  'benchmark_test_positive_ratio',
  'n_features',
  'n_samples',
  'name',
  'original_positive_ratio',
  'test_anomalies',
  'test_normals',
  'train_normals',
  'y_test'],
 'kdd_keys': ['X_test',
  'X_train',
  'attack_types',
  'benchmark_test_positive_ratio',
  'n_features',
  'n_samples',
  'name',
  'original_positive_ratio',
  'test_anomalies',
  'test_normals',
  'train_normals',
  'y_test'],
 'har_keys': ['X_test',
  'X_train',
  'anomaly_activities',
  'benchmark_test_positive_ratio',
  'n_features',
  'n_samples',
  'name',
  'normal_activities',
  'original_positive_ratio',
  'test_anomalies',
  'test_normals',
  'train_normals',
  'y_test']}

### Time Series Datasets

Use load_from_zip_url: Loads a specific file from a ZIP archive hosted at a given URL, file type csv.

If the file inside the ZIP is another ZIP, it will extract that and load the file inside it.


In [3]:
from RADAR.time_series.time_series_datasets_uci import global_load as load_time_series

In [4]:
df_time_series = load_time_series('MetroPT-3')
df_time_series

,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
10,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
20,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
30,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
40,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15169430,2020-09-01 03:59:10,-0.014,8.918,8.906,-0.022,8.918,59.675,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
15169440,2020-09-01 03:59:20,-0.014,8.904,8.888,-0.020,8.904,59.600,0.0450,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
15169450,2020-09-01 03:59:30,-0.014,8.890,8.876,-0.022,8.892,59.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
15169460,2020-09-01 03:59:40,-0.012,8.876,8.864,-0.022,8.878,59.550,0.0450,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


Use a specific function to load the Gas Sensor Temperature Modulation data set:
Loads multiple CSV files from a nested ZIP file hosted at a given URL,  and combines them into a single DataFrame, preserving the date and time of each file.

In [6]:
df_time_series = load_time_series('gas_sensor_temperature_modulation')
df_time_series

,timestamp,Time (s),CO (ppm),Humidity (%r.h.),Temperature (C),Flow rate (mL/min),Heater voltage (V),R1 (MOhm),R2 (MOhm),R3 (MOhm),...,R5 (MOhm),R6 (MOhm),R7 (MOhm),R8 (MOhm),R9 (MOhm),R10 (MOhm),R11 (MOhm),R12 (MOhm),R13 (MOhm),R14 (MOhm)
0,2016-09-30 20:37:18,0.000,0.0,49.7534,23.7184,233.2737,0.8993,0.2231,0.6365,1.1493,...,1.2534,1.4449,1.9906,1.3303,1.4480,1.9148,3.4651,5.2144,6.5806,8.6385
1,2016-09-30 20:37:18,0.309,0.0,55.8400,26.6200,241.6323,0.2112,2.1314,5.3552,9.7569,...,9.4472,10.5769,13.6317,21.9829,16.1902,24.2780,31.1014,34.7193,31.7505,41.9167
2,2016-09-30 20:37:18,0.618,0.0,55.8400,26.6200,241.3888,0.2070,10.5318,22.5612,37.2635,...,33.0704,36.3160,42.5746,49.7495,31.7533,57.7289,53.6275,56.9212,47.8255,62.9436
3,2016-09-30 20:37:18,0.926,0.0,55.8400,26.6200,241.1461,0.2042,29.5749,49.5111,65.6318,...,58.3847,67.5130,68.0064,59.2824,36.7821,66.0832,66.8349,66.9695,50.3730,64.8363
4,2016-09-30 20:37:18,1.234,0.0,55.8400,26.6200,240.9121,0.2030,49.5111,67.0368,77.8317,...,71.7732,79.9474,79.8631,62.5385,39.6271,68.1441,62.0947,49.4614,52.8453,66.8445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3843155,2016-10-16 05:36:56,90908.534,0.0,63.9400,24.6200,0.0000,0.2080,9.9322,19.1578,30.1225,...,28.2241,29.2848,35.2025,54.4724,51.7813,59.3023,63.4039,61.3484,53.8044,65.2822
3843156,2016-10-16 05:36:56,90908.843,0.0,63.9400,24.6200,0.0000,0.2050,31.7887,50.6638,62.3092,...,54.3512,57.2118,64.5292,78.2372,63.7388,69.2228,68.3506,70.6909,62.4181,74.9537
3843157,2016-10-16 05:36:56,90909.151,0.0,63.9400,24.6200,0.0000,0.2040,57.7304,76.9383,80.6932,...,82.5354,74.2258,77.0531,53.0355,70.6869,75.8734,76.1033,71.3160,61.0589,73.6785
3843158,2016-10-16 05:36:56,90909.461,0.0,63.9400,24.6200,0.0000,0.2020,71.9176,82.0040,86.3116,...,79.5571,83.9107,86.1372,82.1159,67.2807,71.4844,68.3506,74.2698,62.4181,74.9537


In [13]:
X,y = load_time_series('metro_interstate_traffic_volume')   #name dataset , y None

Metadata: {'uci_id': 492, 'name': 'Metro Interstate Traffic Volume', 'repository_url': 'https://archive.ics.uci.edu/dataset/492/metro+interstate+traffic+volume', 'data_url': 'https://archive.ics.uci.edu/static/public/492/data.csv', 'abstract': 'Hourly Minneapolis-St Paul, MN traffic volume for westbound I-94. Includes weather and holiday features from 2012-2018.', 'area': 'Other', 'tasks': ['Regression'], 'characteristics': ['Multivariate', 'Sequential', 'Time-Series'], 'num_instances': 48204, 'num_features': 8, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['traffic_volume'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2019, 'last_updated': 'Fri Mar 15 2024', 'dataset_doi': '10.24432/C5X60B', 'creators': ['John Hogue'], 'intro_paper': None, 'additional_info': {'summary': 'Hourly Interstate 94 Westbound traffic volume for MN DoT ATR station 301, roughly midway between Minneapolis and St Paul, MN. Ho

In [14]:
X.drop(["date_time", "holiday","weather_main","weather_description"], axis=1)

,temp,rain_1h,snow_1h,clouds_all
0,288.28,0.0,0.0,40
1,289.36,0.0,0.0,75
2,289.58,0.0,0.0,90
3,290.13,0.0,0.0,90
4,291.14,0.0,0.0,75
...,...,...,...,...
48199,283.45,0.0,0.0,75
48200,282.76,0.0,0.0,90
48201,282.73,0.0,0.0,90
48202,282.09,0.0,0.0,90


### Preprocessing

In [6]:
from RADAR.time_series.time_series_datasets_uci import global_load as load_time_series
from RADAR.static_data.preprocessing.preprocessing_static import OneHotEncoderPreprocessing,StandardScalerPreprocessing 


##### General example One Hot Encode

In [7]:
import pandas as pd
data = pd.DataFrame({
    'color': ['rojo', 'azul', 'verde', 'rojo', 'azul'],
    'tamaño': ['grande', 'pequeño', 'mediano', 'grande', 'mediano'],
    'precio': [10, 15, 20, 25, 30]
})

encoder = OneHotEncoderPreprocessing(columns=['color', 'tamaño'])


data_encoded = encoder.fit_transform(data)

print("Data después de la codificación:")
print(data_encoded)

data_decoded = encoder.inverse_transform(data_encoded)
print("\nData después de la inversión:")
print(data_decoded)

Data después de la codificación:
   precio  color_azul  color_rojo  color_verde  tamaño_grande  tamaño_mediano  \
0      10           0           1            0              1               0   
1      15           1           0            0              0               0   
2      20           0           0            1              0               1   
3      25           0           1            0              1               0   
4      30           1           0            0              0               1   

   tamaño_pequeño  
0               0  
1               1  
2               0  
3               0  
4               0  

Data después de la inversión:
   precio  color   tamaño
0      10   rojo   grande
1      15   azul  pequeño
2      20  verde  mediano
3      25   rojo   grande
4      30   azul  mediano


##### Example of preprocessing with data sets from the Uci repository

In [ ]:
X,y = load_time_series('ai4i_2020_predictive_maintenance_dataset')   #name dataset in static datasets uci repo

In [ ]:
print(X["Type"].unique())

In [30]:
encoder = OneHotEncoderPreprocessing(columns=['Type'])
X_encoded = encoder.fit_transform(X)

print("Data after encoding:")
print(X_encoded)

X_decoded = encoder.inverse_transform(X_encoded)
print("\nData after reversal:")
print(X_decoded)

Data after encoding:
      Air temperature  Process temperature  Rotational speed  Torque  \
0               298.1                308.6              1551    42.8   
1               298.2                308.7              1408    46.3   
2               298.1                308.5              1498    49.4   
3               298.2                308.6              1433    39.5   
4               298.2                308.7              1408    40.0   
...               ...                  ...               ...     ...   
9995            298.8                308.4              1604    29.5   
9996            298.9                308.4              1632    31.8   
9997            299.0                308.6              1645    33.4   
9998            299.0                308.7              1408    48.5   
9999            299.0                308.7              1500    40.2   

      Tool wear  Type_H  Type_L  Type_M  
0             0       0       0       1  
1             3       0       

In [35]:
scaler = StandardScalerPreprocessing()
numerical_cols = X.select_dtypes(include=['float64', 'int64']).columns
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])
X_encoded




,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Type_H,Type_L,Type_M
0,-0.952389,-0.947360,0.068185,0.282200,-1.695984,0,0,1
1,-0.902393,-0.879959,-0.729472,0.633308,-1.648852,0,1,0
2,-0.952389,-1.014761,-0.227450,0.944290,-1.617430,0,1,0
3,-0.902393,-0.947360,-0.590021,-0.048845,-1.586009,0,1,0
4,-0.902393,-0.879959,-0.729472,0.001313,-1.554588,0,1,0
...,...,...,...,...,...,...,...,...
9995,-0.602417,-1.082162,0.363820,-1.052012,-1.476034,0,0,1
9996,-0.552421,-1.082162,0.520005,-0.821283,-1.428902,1,0,0
9997,-0.502425,-0.947360,0.592519,-0.660777,-1.350349,0,0,1
9998,-0.502425,-0.879959,-0.729472,0.854005,-1.303217,1,0,0


In [ ]:
print(f"X:{X_encoded}, y:{y}")